In [2]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

file_path = r"cw1 -dataset.xlsx"

xls = pd.ExcelFile(file_path)

bmi = pd.read_excel(xls, 'BMI')
bp = pd.read_excel(xls, 'Raised Blood Pressure')
diabetes = pd.read_excel(xls, 'Diabetes')

bmi = bmi.rename(columns={
    "Country/Region/World": "Country",
    "Sex": "Gender",
    "Prevalence of BMI>=30 kg/m≤ (obesity)": "BMI"
})

bp = bp.rename(columns={
    "Country/Region/World": "Country",
    "Sex": "Gender",
    "Prevalence of raised blood pressure": "BP"
})

diabetes = diabetes.rename(columns={
    "Country/Region/World": "Country",
    "Sex": "Gender",
    "Age-standardised diabetes prevalence": "Diabetes"
})

print("Column Names: ", diabetes.columns)

data = bmi.merge(bp, on=["Country", "Year", "Gender"], how="outer")
data = data.merge(diabetes, on=["Country", "Year", "Gender"], how="outer")

data = data.dropna(subset=["BMI", "BP", "Diabetes"])

data['Year'] = data['Year'].astype(int)
data['BMI'] = pd.to_numeric(data['BMI'], errors='coerce')
data['BP'] = pd.to_numeric(data['BP'], errors='coerce')
data['Diabetes'] = pd.to_numeric(data['Diabetes'], errors='coerce')

data = data.dropna(subset=["BMI", "BP", "Diabetes"])

data['BMI'] *= 100
data['BP'] *= 100
data['Diabetes'] *= 100

gdp = pd.read_csv("GDP.csv")
income = pd.read_csv("GDP Metadata.csv")
urban_population = pd.read_csv("Urban Population.csv")

gdp = gdp.drop(columns=['GDP per capita (Annotations)'], errors='ignore')
gdp_long = gdp.rename(columns={'Entity': 'Country', 'Code': 'Country Code', 'GDP per capita': 'GDP'})
gdp_long = gdp_long[~gdp_long['Country Code'].str.contains('AFE|AFW', na=False)]
gdp_long['Year'] = pd.to_numeric(gdp_long['Year'], errors='coerce')
gdp_long = gdp_long[(gdp_long['Year'] >= 1980) & (gdp_long['Year'] <= 2016)]

urban_population.rename(columns={
    urban_population.columns[0]: 'Country Name',
    urban_population.columns[1]: 'Country Code'
}, inplace=True)

urban_population = urban_population.drop(
    columns=['Indicator Name', 'Indicator Code', 'Unnamed: 69'],
    errors='ignore'
)
urban_long = urban_population.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='Urban_Population'
)
urban_long['Year'] = pd.to_numeric(urban_long['Year'], errors='coerce')
urban_long = urban_long[(urban_long['Year'] >= 1980) & (urban_long['Year'] <= 2016)]
urban_long = urban_long.rename(columns={'Country Name': 'Country'})
urban_long.columns

income_clean = income[['Country Code', 'Region', 'IncomeGroup']]

health_countries = set(data['Country'].unique())
urban_countries = set(urban_long['Country'].unique())

missing_in_urban = health_countries - urban_countries
print("Countries in health data not in urban population:", missing_in_urban)

urban_long = urban_population.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='Urban_Population'
)
urban_long = urban_long.rename(columns={'Country Name': 'Country'})
urban_long['Year'] = pd.to_numeric(urban_long['Year'], errors='coerce')

data = data.merge(
    urban_long[['Country', 'Year', 'Urban_Population']],
    on=['Country', 'Year'],
    how='left'
)

print("Missing Urban Population values:", data['Urban_Population'].isna().sum())
print(data[data['Urban_Population'].isna()][['Country', 'Year']].drop_duplicates())

code_to_country = urban_long[['Country Code', 'Country']].drop_duplicates()


urban_long_clean = urban_long[['Country', 'Year', 'Urban_Population']].copy()
data = data.merge(
    urban_long[['Country', 'Year', 'Urban_Population']],
    on=['Country', 'Year'],
    how='left'
)

if 'Urban_Population_x' in data.columns:
    data['Urban_Population'] = data['Urban_Population_x']
    data = data.drop(columns=['Urban_Population_x', 'Urban_Population_y'], errors='ignore')

print("Columns in health data after adding Urban Population:", data.columns)
print("Missing Urban Population values:", data['Urban_Population'].isna().sum())

print("Columns in health data after adding Urban Population:", data.columns)
print("Missing Urban Population values:", data['Urban_Population'].isna().sum())
print(data[data['Urban_Population'].isna()][['Country', 'Year']].drop_duplicates())

income_clean = income[['Country Code', 'Region', 'IncomeGroup']].copy()
income_clean = income_clean.merge(
    code_to_country,  
    on='Country Code',
    how='left'
)
print("Income sample after merge:")
print(income_clean[['Country Code', 'Country', 'Region', 'IncomeGroup']].head())
gdp_long = gdp_long[~gdp_long['Country Code'].str.contains('AFE|AFW|AFR|EAS|OCE', na=False)]

gdp_long['Country Code'] = gdp_long['Country Code'].str.strip()
code_to_country['Country Code'] = code_to_country['Country Code'].str.strip()

gdp_long = gdp_long.merge(
    code_to_country[['Country Code', 'Country']], 
    on='Country Code', 
    how='left'
)
if 'Country_y' in gdp_long.columns:
    gdp_long['Country'] = gdp_long['Country_y']
    gdp_long = gdp_long.drop(columns=['Country_x', 'Country_y'], errors='ignore')

gdp_long.loc[gdp_long['Country Code']=='TWN', 'Country'] = 'Taiwan'

print(gdp_long[['Country Code', 'Country', 'Year', 'GDP']].head())

missing_gdp = gdp_long[gdp_long['Country'].isna()]['Country Code'].unique()
missing_income = income_clean[income_clean['Country'].isna()]['Country Code'].unique()

print("Missing GDP countries:", missing_gdp)
print("Missing Income countries:", missing_income)

data = data.merge(
    gdp_long[['Country', 'Year', 'GDP']],
    on=['Country', 'Year'],
    how='left'
)

data = data.merge(
    income_clean[['Country', 'Region', 'IncomeGroup']],
    on='Country',
    how='left'
)

if 'Urban_Population_x' in data.columns:
    data['Urban_Population'] = data['Urban_Population_x']
    data = data.drop(columns=['Urban_Population_x', 'Urban_Population_y'], errors='ignore')

print("Columns in final master dataframe:")
print(data.columns)
print("Sample rows:")
print(data.head())

print("Missing values per column:")
print(data[['BMI', 'BP', 'Diabetes', 'Urban_Population', 'GDP', 'Region', 'IncomeGroup']].isna().sum())

data['IncomeGroup'] = data['IncomeGroup'].fillna("Not Classified")
print(data.isna().sum().sort_values(ascending=False))

missing_gdp_counts = (
    data[data['GDP'].isna()]
    .groupby('Country')
    .size()
    .sort_values(ascending=False)
)

print(missing_gdp_counts.head(50))

print(data[(data['Country'] == 'United Arab Emirates') & (data['GDP'].isna())][['Year', 'Gender']])
print(gdp_long[gdp_long['Country'] == 'United Arab Emirates'][['Year', 'GDP']].sort_values('Year'))

data = data.dropna(subset=['GDP', "Region", "Urban_Population"])

print(data.isna().sum().sort_values(ascending=False))

print(data.columns)

Column Names:  Index(['Country', 'Gender', 'Year', 'Diabetes'], dtype='object')
Countries in health data not in urban population: {'Taiwan', 'Niue', 'Tokelau', 'Cook Islands'}
Missing Urban Population values: 280
           Country  Year
1470  Cook Islands  1980
1471  Cook Islands  1981
1472  Cook Islands  1982
1473  Cook Islands  1983
1474  Cook Islands  1984
...            ...   ...
6330       Tokelau  2010
6331       Tokelau  2011
6332       Tokelau  2012
6333       Tokelau  2013
6334       Tokelau  2014

[140 rows x 2 columns]
Columns in health data after adding Urban Population: Index(['Country', 'Gender', 'Year', 'BMI', 'BP', 'Diabetes',
       'Urban_Population'],
      dtype='object')
Missing Urban Population values: 280
Columns in health data after adding Urban Population: Index(['Country', 'Gender', 'Year', 'BMI', 'BP', 'Diabetes',
       'Urban_Population'],
      dtype='object')
Missing Urban Population values: 280
           Country  Year
1470  Cook Islands  1980
1471  Coo

In [3]:
@alt.theme.register('cyberpunk_dark', enable=True)
def cyberpunk_theme():
    return alt.theme.ThemeConfig({
        'config': {
            'background': '#1e1e26',  # Deep grey-blue like your crypto example
            'view': {
                'stroke': 'transparent',
                'fill': '#1e1e26'
            },
            'axis': {
                'domainColor': '#FFFFFF', 
                'gridColor': '#333333',   # Subtle grid like the marketing dash
                'labelColor': '#FFFFFF',
                'titleColor': '#FFFFFF',
                'tickColor': '#FFFFFF',
                'gridOpacity': 0.5
            },
            'legend': {
                'labelColor': '#FFFFFF',
                'titleColor': '#FFFFFF',
                'padding': 10,
                'labelFontSize': 11
            },
            'title': {
                'color': '#FFFFFF',
                'fontSize': 20,
                'anchor': 'start', # Professional left-aligned titles
                'font': 'sans-serif'
            },
            'range': {
                'category': ['#00D4FF', '#FF007F', '#32CD32', '#FFD700', '#FF8C00', '#8A2BE2'], # Neon palette
            },
            'tooltip': {'theme': 'dark'} # Keeps the vibe consistent
        }
    })

In [9]:
# 1. Region Dropdown with an "All" option
region_options = [None] + sorted(data['Region'].dropna().unique().tolist())
region_labels = ['Global (All)'] + sorted(data['Region'].dropna().unique().tolist())

select_region = alt.selection_point(
    fields=['Region'], 
    bind=alt.binding_select(options=region_options, labels=region_labels, name='Select Region: '),
    name="RegionSelector",
    value=None
)

# 2. Year Slider (1980 - 2016)
year_slider = alt.selection_point(
    fields=['Year'], 
    bind=alt.binding_range(min=1980, max=2016, step=1, name='Select Year: '),
    name="YearSelector",
    value=2016
)

# Gender scale: Blue for Men, Pink for Women
# Note: Use 'Men'/'Women' or 'Male'/'Female' depending on your data
gender_scale = alt.Scale(domain=['Men', 'Women'], range=['#00D4FF', '#FF007F'])

# Region scale: Bright colors for regional metrics
region_scale = alt.Scale(scheme='tableau10')

In [10]:
scatter = alt.Chart(data).mark_circle(opacity=0.6, 
    stroke='white', 
    strokeWidth=0.5).encode(
    x=alt.X('Urban_Population:Q', title='Urban Population (%)'),
    # Note: You previously mentioned this is Obesity Prevalence (%)
    y=alt.Y('BMI:Q', title='Obesity Prevalence (%)'), 
    color=alt.Color('Region:N', legend=alt.Legend(title="Region")),
    size=alt.Size('GDP:Q', 
                  legend=alt.Legend(title="GDP per Capita", orient='right'), 
                  # Shrinking the max from 1000 to 350
                  scale=alt.Scale(range=[20, 350])) 
).transform_filter(
    select_region & year_slider
)

In [13]:
# 1. The Main Title (A text-based chart to keep it in the dashboard)
title_header = alt.Chart(pd.DataFrame({'t': ["Global Health & Wealth Insights 1980–2016"]})).mark_text(
    align='left', baseline='middle', fontSize=28, fontWeight='bold', color='#00D4FF', dx=-200
).encode(text='t:N').properties(width=800, height=50)

# 2. KPI Card 1: Average Obesity Prevalence (%)
# We use 'mean(BMI)' to calculate based on the current selection
kpi_obesity = alt.Chart(data).mark_text(fontSize=40, fontWeight='bold', color='#FF007F').encode(
    text=alt.Text('mean(BMI):Q', format='.1f')
).transform_filter(
    select_region & year_slider
).properties(width=200, height=100, title="Avg. Obesity %")

# 3. KPI Card 2: Average GDP
kpi_gdp = alt.Chart(data).mark_text(fontSize=40, fontWeight='bold', color='#00D4FF').encode(
    text=alt.Text('mean(GDP):Q', format='$,.0f')
).transform_filter(
    select_region & year_slider
).properties(width=200, height=100, title="Avg. GDP per Capita")

# 4. KPI Card 3: Urbanization Level
kpi_urban = alt.Chart(data).mark_text(fontSize=40, fontWeight='bold', color='#32CD32').encode(
    text=alt.Text('mean(Urban_Population):Q', format='.1f')
).transform_filter(
    select_region & year_slider
).properties(width=200, height=100, title="Avg. Urbanization %")

# --- Combine KPIs into a Horizontal "Command Center" ---
# Removed .resolve_scale(text='independent') to fix the error
kpi_row = alt.hconcat(kpi_obesity, kpi_gdp, kpi_urban)

# Combine with the Title
header_section = alt.vconcat(title_header, kpi_row).configure_view(stroke=None)

In [16]:
line_year_filter = alt.datum.Year <= year_slider.Year

line_obesity = alt.Chart(data).mark_line(point=True).encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('mean(BMI):Q', title='Obesity Prevalence (%)'), 
    color=alt.Color('Gender:N', 
                    scale=gender_scale, 
                    legend=alt.Legend(title="Gender")),
    tooltip=[
        alt.Tooltip('Year:O'),
        alt.Tooltip('Gender:N'),
        alt.Tooltip('mean(BMI):Q', title='Avg. Prevalence (%)', format='.1f')
    ]
).transform_filter(
    select_region # Only show countries in selected region
).transform_filter(
    line_year_filter # Only show years up to the slider value
).properties(
    width=400, 
    height=350, 
    title="Trend of Obesity Prevalence Over Time"
)

In [18]:
# Combine the charts
dashboard = alt.hconcat(scatter, line_obesity).resolve_scale(
    color='independent', 
    size='independent'
).resolve_legend(
    color='independent', 
    size='independent'
).add_params(
    select_region, year_slider
)

# 1. Convert the chart object to an HTML string
# We use a placeholder for the chart to ensure the string is fully rendered
html_string = dashboard.to_html()

# 2. Define your CSS (your "Top Right" styling)
custom_css = """
<style>
  .vega-bindings {
    position: fixed !important;
    top: 20px !important;
    right: 20px !important;
    display: flex;
    flex-direction: column;
    background: #333 !important;
    padding: 15px;
    border-radius: 8px;
    border: 1px solid #555;
    z-index: 1000;
    color: white !important;
  }
  .vega-bind { margin-bottom: 10px; }
  .vega-bind-name { font-weight: bold; color: #00D4FF; }
</style>
"""

# 3. NOW you can use .replace() because 'html_string' is a string
final_html = html_string.replace('</head>', f'{custom_css}</head>')

# 4. Save the final string to a file
with open("Final_Dashboard.html", "w", encoding="utf-8") as f:
    f.write(final_html)

print("Dashboard saved! The filters are now in the top-right corner.")

C:\Users\larai\AppData\Local\Programs\Python\Python311\Lib\site-packages\altair\vegalite\v6\api.py:2266: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  self.to_dict(),


Dashboard saved! The filters are now in the top-right corner.
